In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import joblib
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline

N_FOLDS = 5
RANDOM_STATE = 42
DATA_RAW_DIR = "data/raw"

In [ ]:
def load_and_merge_data():
    print("🔄 MEMULAI PROSES ETL (Hybrid Architecture)...")
    
    def get_latest_file(pattern):
        # Cari file dengan pattern tertentu, ambil yang paling baru dibuat
        files = glob.glob(os.path.join(DATA_RAW_DIR, pattern))
        return max(files, key=os.path.getctime) if files else None

    # 1. Cari File Utama (Inventory Time Series)
    # File ini sudah berisi: Date, Ticker, OHLCV, dan NET_VOL (Bandar Flow)
    inv_path = get_latest_file("inventory_ts_*.xlsx")
    
    # 2. Cari File Shareholder (Masih terpisah)
    sh_path = get_latest_file("shareholder_*.xlsx")

    if not inv_path:
        print("❌ Error: File 'inventory_ts_*.xlsx' tidak ditemukan. Jalankan main.py dulu!")
        return None

    print(f"📂 Loading Main Data: {os.path.basename(inv_path)}")
    df_inv = pd.read_excel(inv_path)
    
    # Standarisasi kolom
    df_inv.columns = df_inv.columns.str.strip().str.lower()
    df_inv['date'] = pd.to_datetime(df_inv['date']) # Pastikan datetime

    # 3. Merge dengan Shareholder Data (Jika ada)
    if sh_path:
        print(f"📂 Loading Shareholder: {os.path.basename(sh_path)}")
        df_sh = pd.read_excel(sh_path)
        df_sh.columns = df_sh.columns.str.strip().str.lower()
        df_sh['date'] = pd.to_datetime(df_sh['date'])
        
        # Merge (Left Join)
        merged_df = pd.merge(df_inv, df_sh[['ticker', 'date', 'value']], on=['ticker', 'date'], how='left')
        
        # Forward Fill Shareholder Count (Karena datanya bulanan)
        # Artinya: Jika hari ini tidak ada update, pakai data bulan lalu
        merged_df['shareholder_count'] = merged_df.groupby('ticker')['value'].ffill()
        
        # Hapus kolom bantuan 'value' agar tidak bingung
        merged_df = merged_df.drop(columns=['value'])
    else:
        print("⚠️ Warning: File Shareholder tidak ada. Fitur retail_expansion akan kosong.")
        merged_df = df_inv
        merged_df['shareholder_count'] = np.nan

    print(f"🤝 Total Data Siap Olah: {len(merged_df)} baris.")
    return merged_df

# Load Data
df_raw = load_and_merge_data()

In [ ]:
def create_features(df):
    print("🐳 MEMBUAT FITUR: WHALE TRACKING & VWAP ESTIMATION...")
    df = df.sort_values(by=['ticker', 'date']).reset_index(drop=True)

    # 1. Cleaning Dasar
    df['volume'] = df['volume'].fillna(0)
    df['net_vol'] = df['net_vol'].fillna(0) # Ini data bandar dari Invezgo
    
    # 2. Fitur Dominasi Bandar (Bandarmology)
    # Range: -1 (Total Distribusi) s/d 1 (Total Akumulasi)
    df['inventory_intensity'] = df['net_vol'] / (df['volume'] + 1)
    
    # Konsistensi Akumulasi (5 Hari)
    df['inventory_velocity'] = df.groupby('ticker')['net_vol'].transform(lambda x: x.rolling(5).sum())

    # 3. Estimasi Modal Bandar (VWAP Logic) - OTAK MODEL KITA
    # Hitung Typical Price
    df['typical_price'] = (df['high'] + df['low'] + df['close']) / 3
    
    # Hanya hitung saat Bandar Beli (Uang Masuk)
    df['whale_buy_vol'] = np.where(df['net_vol'] > 0, df['net_vol'], 0)
    df['whale_buy_val'] = df['whale_buy_vol'] * df['typical_price']
    
    # Rolling 20 Hari (Asumsi siklus pendek bandar)
    roll_buy_val = df.groupby('ticker')['whale_buy_val'].transform(lambda x: x.rolling(20).sum())
    roll_buy_vol = df.groupby('ticker')['whale_buy_vol'].transform(lambda x: x.rolling(20).sum())
    
    # Hitung VWAP (Harga Rata-rata Modal)
    df['est_whale_cost'] = np.where(roll_buy_vol > 0, roll_buy_val / roll_buy_vol, df['close'])
    
    # 4. Cost Gap (Golden Feature)
    # Negatif = Diskon (BUY ZONE), Positif = Mahal (SELL ZONE)
    df['est_cost_gap'] = (df['close'] - df['est_whale_cost']) / (df['est_whale_cost'] + 0.0001)

    # 5. Fitur Konteks & Target
    # Retail Expansion (Jika ada data shareholder)
    if 'shareholder_count' in df.columns:
        df['sh_change'] = df.groupby('ticker')['shareholder_count'].diff()
        df['retail_expansion'] = np.where(df['sh_change'] > 0, 1, 0)
    else:
        df['retail_expansion'] = 0

    # Technical Context
    df['ma20'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(20).mean())
    df['trend_position'] = df['close'] / (df['ma20'] + 0.0001)
    df['rvol'] = df['volume'] / (df.groupby('ticker')['volume'].transform(lambda x: x.rolling(20).mean()) + 1)
    df['candle_spread'] = (df['high'] - df['low']) / (df['open'] + 0.1)

    # Target (Next Close > 1%)
    df['next_close'] = df.groupby('ticker')['close'].shift(-1)
    df['target_class'] = np.where(df['next_close'] > df['close'] * 1.01, 1, 0)
    
    # Cleanup NaN
    features_to_check = ['est_whale_cost', 'ma20', 'next_close']
    final_df = df.dropna(subset=features_to_check)
    
    return final_df


# Eksekusi
if df_raw is not None:
    final_df = create_features(df_raw)
    
    # Cek Kolom Penting
    print(f"\n✅ Preprocessing Selesai: {len(final_df)} baris.")
    print("Contoh Fitur Baru (est_cost_gap):")
    print(final_df[['date', 'ticker', 'close', 'net_vol', 'est_whale_cost', 'est_cost_gap']].tail())

In [ ]:
feature_cols = [
    # --- 1. Bandarmology Core (Kekuatan & Arah) ---
    'net_vol',              # Volume bersih (Raw)
    'inventory_intensity',  # [BARU] Seberapa dominan bandar hari ini (-1 s/d 1)
    'inventory_velocity',   # [PENTING] Konsistensi akumulasi 5 hari terakhir
    
    # --- 2. Whale Valuation (Posisi Harga vs Modal) ---
    'est_cost_gap',         # [GOLDEN FEATURE] Jarak harga Close vs Estimasi Modal Bandar (VWAP)
    
    # --- 3. Retail Sentiment (Contrarian) ---
    'sh_change',            # Perubahan jumlah pemegang saham
    'retail_expansion',     # Flag: 1 jika ritel masuk (Bad), 0 jika ritel keluar (Good)
    
    # --- 4. Technical Context (Momentum) ---
    'trend_position',       # Posisi harga terhadap MA20 (Uptrend/Downtrend)
    'rvol',                 # Relative Volume (Ledakan volume vs Rata-rata)
    'candle_spread'         # Volatilitas candle (High - Low)
]

X = final_df[feature_cols].values
y = final_df['target_class'].values

# Penyeimbangan (Imbalance Handling)
neg, pos = (y == 0).sum(), (y == 1).sum()
scale_pos_weight = neg / pos if pos > 0 else 1.0
print(f"⚖️ Scale Pos Weight: {scale_pos_weight:.2f}")

In [ ]:
import optuna

def objective(trial):
    # --- Hyperparameter Search Space ---
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'random_state': RANDOM_STATE,
        'scale_pos_weight': scale_pos_weight,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
    }

    # Cross-Validation di dalam Optuna
    scores = []
    kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE) # 3-fold untuk speed tuning
    
    for tr_idx, va_idx in kf.split(X, y):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        
        dtrain = lgb.Dataset(X_tr, y_tr)
        lgb_m = lgb.train(params, dtrain, num_boost_round=200)
        
        preds = (lgb_m.predict(X_va) > 0.5).astype(int)
        score = f1_score(y_va, preds, average='macro')
        scores.append(score)
    
    return np.mean(scores)

# --- 1. Jalankan Studi Optuna ---
print("🔍 Mencari Hyperparameter Terbaik dengan Optuna...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50) # Ubah n_trials sesuai kekuatan PC Anda

print(f"✅ Best Trial Score: {study.best_value:.4f}")
print(f"📌 Best Params: {study.best_params}")

In [ ]:
# --- 2. Re-Train dengan Parameter Terbaik (Full Pipeline) ---
best_lgb_params = {**study.best_params, 'objective': 'binary', 'verbosity': -1, 'scale_pos_weight': scale_pos_weight}

oof_lgb = np.zeros(len(X))
oof_rf = np.zeros(len(X))
kf_final = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

for fold, (tr_idx, va_idx) in enumerate(kf_final.split(X, y), 1):
    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    # Re-Train LightGBM dengan Best Params
    dtrain = lgb.Dataset(X_tr, y_tr)
    lgb_m = lgb.train(best_lgb_params, dtrain, num_boost_round=500)
    oof_lgb[va_idx] = lgb_m.predict(X_va)

    # Random Forest tetap (atau bisa ditambahkan Optuna juga jika mau)
    rf_m = RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=-1)
    rf_m.fit(X_tr, y_tr)
    oof_rf[va_idx] = rf_m.predict_proba(X_va)[:, 1]
    
    print(f"✅ Fold {fold} Final Selesai.")

In [ ]:
import optuna.visualization as vis

# --- 1. Plot History Optimasi ---
# Menunjukkan bagaimana F1-Score meningkat seiring bertambahnya trial
print("📈 Menampilkan Riwayat Optimasi...")
fig_history = vis.plot_optimization_history(study)
fig_history.show()

# --- 2. Plot Parameter Importance ---
# Menunjukkan hyperparameter mana yang paling berpengaruh terhadap akurasi
print("📊 Menampilkan Kepentingan Hyperparameter...")
fig_importance = vis.plot_param_importances(study)
fig_importance.show()

# --- 3. Plot Slice ---
# Menunjukkan sebaran nilai parameter yang dicoba
print("🍕 Menampilkan Sebaran Parameter (Slice Plot)...")
fig_slice = vis.plot_slice(study, params=['learning_rate', 'num_leaves', 'max_depth'])
fig_slice.show()

In [ ]:
# Blending sederhana untuk display
final_prob = (0.5 * oof_lgb) + (0.5 * oof_rf)
y_pred = (final_prob >= 0.5).astype(int)

# Visualisasi
plt.figure(figsize=(15, 5))

# 1. Confusion Matrix
plt.subplot(1, 2, 1)
sns.heatmap(confusion_matrix(y, y_pred), annot=True, fmt='d', cmap='Greens')
plt.title('Confusion Matrix')

# 2. Feature Importance
plt.subplot(1, 2, 2)
importance = lgb_m.feature_importance(importance_type='gain')
pd.Series(importance, index=feature_cols).sort_values().plot(kind='barh')
plt.title('Fitur Paling Berpengaruh (Invezgo Features)')

plt.tight_layout()
plt.show()

# Simpan Model Final
joblib.dump({'lgb': lgb_m, 'rf': rf_m, 'features': feature_cols}, "models/final_blend_model.pkl")

In [ ]:
# --- Sel Akhir: Rekomendasi Sinyal Tertinggi ---
def show_top_signals(df, prob_array, top_n=5):
    print(f"\n🏆 TOP {top_n} STOCK SIGNALS (BASED ON AI TRAINING)")
    print("====================================================")
    
    # Ambil baris terakhir untuk setiap ticker (data terbaru)
    # Kita lampirkan hasil probabilitas ke dataframe
    df_results = df.copy()
    df_results['ai_probability'] = prob_array
    
    # Ambil data terbaru per ticker
    latest_signals = df_results.groupby('ticker').tail(1)
    
    # Urutkan berdasarkan probabilitas tertinggi
    top_signals = latest_signals.sort_values(by='ai_probability', ascending=False).head(top_n)
    
    # Tampilkan kolom kunci untuk analisis cepat
    display_cols = [
        'ticker', 'close', 'ai_probability', 
        'trend_position', 'avg_lot_size', 'retail_expansion'
    ]
    
    summary = top_signals[display_cols].reset_index(drop=True)
    
    # Memberikan interpretasi sederhana
    def interpret_signal(row):
        if row['ai_probability'] >= 0.6 and row['trend_position'] > 1:
            return "🔥 STRONG ACCUMULATION & MARKUP"
        elif row['ai_probability'] >= 0.5:
            return "✅ ACCUMULATION DETECTED"
        else:
            return "⏳ MONITORING"

    summary['status'] = summary.apply(interpret_signal, axis=1)
    
    # Styling DataFrame agar lebih cantik di Notebook
    return summary.style.background_gradient(subset=['ai_probability'], cmap='RdYlGn')

# Panggil fungsi
show_top_signals(final_df, final_prob)